In [9]:
from src.feature_selection import *
from src.helpers import *
from src.metamorphic_tests import run_metamorphic_tests
from src.models import *
from src.partition_tests import run_partition_tests
from src.reweighting import *
from sklearn.model_selection import train_test_split

 # Data Loading and Inspection

In [10]:
DATA_PATH = "data/investigation_train_large_checked.csv"
ADDITIONAL_DATA_PATH = "data/additional_synth_data.csv"

## Dataset provided from assignment (130k rows)

In [11]:
df = pd.read_csv(DATA_PATH)
print("Data shape (provided):", df.shape)
df.head()

Data shape (provided): (130000, 318)


,adres_aantal_brp_adres,adres_aantal_verschillende_wijken,adres_aantal_verzendadres,adres_aantal_woonadres_handmatig,adres_dagen_op_adres,adres_recentst_onderdeel_rdam,adres_recentste_buurt_groot_ijsselmonde,adres_recentste_buurt_nieuwe_westen,adres_recentste_buurt_other,adres_recentste_buurt_oude_noorden,...,typering_hist_ind,typering_hist_sector_zorg,typering_ind,typering_indicatie_geheime_gegevens,typering_other,typering_transport__logistiek___tuinbouw,typering_zorg__schoonmaak___welzijn,Ja,Nee,checked
0,1,1,0,0,23240,1,0,0,0,0,...,1,0,0,0,0,0,0,0.617698,0.382302,False
1,4,2,1,1,1971,1,0,0,1,0,...,1,0,1,0,1,0,0,0.602167,0.397833,False
2,6,4,2,1,7247,0,0,0,1,0,...,1,0,1,0,0,0,0,0.512377,0.487623,False
3,3,2,0,1,8060,1,0,0,1,0,...,1,0,0,0,0,0,0,0.717796,0.282204,True
4,3,2,0,0,18705,1,0,0,0,0,...,1,0,1,0,0,0,0,0.705484,0.294516,True


## Additional synthetic dataset (30k rows)

In [12]:
df_additional = pd.read_csv(ADDITIONAL_DATA_PATH)       # Generated using: https://github.com/abuszydlik/Social-Welfare-Dataset/blob/main/DataManual.md
print("Data shape (additional):", df_additional.shape)
df_additional.head()

Data shape (additional): (30000, 315)


,adres_aantal_brp_adres,adres_aantal_verschillende_wijken,adres_aantal_verzendadres,adres_aantal_woonadres_handmatig,adres_dagen_op_adres,adres_recentst_onderdeel_rdam,adres_recentste_buurt_groot_ijsselmonde,adres_recentste_buurt_nieuwe_westen,adres_recentste_buurt_other,adres_recentste_buurt_oude_noorden,...,typering_dagen_som,typering_hist_aantal,typering_hist_inburgeringsbehoeftig,typering_hist_ind,typering_hist_sector_zorg,typering_ind,typering_indicatie_geheime_gegevens,typering_other,typering_transport__logistiek___tuinbouw,typering_zorg__schoonmaak___welzijn
0,2,2,2,0,10260,1,0,0,1,0,...,2835,2,0,1,0,1,0,0,0,0
1,1,2,1,1,17654,1,0,0,1,0,...,3717,1,0,1,0,0,0,1,0,0
2,2,2,0,1,7114,1,0,0,1,0,...,1043,2,0,1,0,1,0,1,0,0
3,4,3,1,1,9426,1,0,0,0,0,...,5409,1,0,1,0,0,0,0,0,0
4,2,3,0,1,5972,1,0,0,1,0,...,4675,2,0,1,0,1,0,0,0,0


## Defining target and leakage columns


In [13]:
y = df["checked"].astype(int)
leakage_cols = ["checked", "Ja", "Nee"]
X = df.drop(columns=leakage_cols, errors="ignore")
X_additional = df_additional.drop(columns=leakage_cols, errors="ignore")

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (130000, 315)
Shape y: (130000,)


## All features

In [14]:
cols = set(df.columns)
print(cols)

{'afspraak_laatstejaar_aantal_woorden', 'contacten_soort_afgelopenjaar_rapportage_rib', 'contacten_onderwerp_boolean_no_show', 'contacten_onderwerp_vakantie', 'adres_recentste_wijk_kralingen_c', 'instrument_ladder_huidig_werk_re_integratie', 'contacten_onderwerp_uitnodiging', 'persoonlijke_eigenschappen_nl_schrijven2', 'afspraak_deelname_compleet_uit_webapplicatie', 'persoonlijke_eigenschappen_houding_opm', 'adres_recentste_buurt_other', 'belemmering_dagen_financiele_problemen', 'persoon_geslacht_vrouw', 'persoonlijke_eigenschappen_nl_spreken1', 'relatie_overig_actueel_vorm__gemachtigde', 'afspraak_afgelopen_jaar_plan_van_aanpak', 'afspraak_signaal_van_aanbieder', 'pla_einde_doelstelling_bereikt__nieuw_trajectplan', 'contacten_soort_afgelopenjaar_e_mail__inkomend_', 'contacten_onderwerp_boolean__pre__intake', 'contacten_onderwerp_traject', 'relatie_overig_bewindvoerder', 'contacten_onderwerp_overige', 'typering_hist_aantal', 'instrument_reden_beeindiging_historie_succesvol', 'contacten

## Top features correlated to the target

In [15]:
corr_df = get_target_correlation(X, y)
print("Top features correlated with target:")
print(corr_df.head(30))

Top features correlated with target:
                                                    correlation
relatie_overig_actueel_vorm__kostendeler               0.199893
relatie_overig_kostendeler                             0.177055
contacten_onderwerp_no_show                            0.175866
afspraak_resultaat_ingevuld_uniek                      0.163221
contacten_onderwerp_overleg_met_inkomen                0.154653
contacten_soort_afgelopenjaar_document__uitgaand_      0.145525
competentie_vakdeskundigheid_toepassen                 0.139934
relatie_overig_historie_vorm__kostendeler              0.138380
afspraak_inspanningsperiode                            0.131487
contacten_onderwerp_boolean_no_show                    0.125044
contacten_onderwerp_inspanningstoets                   0.121893
contacten_onderwerp__werk_intake                       0.121008
deelname_act_reintegratieladder_werk_re_integratie     0.118024
afspraak_controle_aankondiging_maatregel               0.116746
con

# Model Training

## Splitting data into train and test sets

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X.fillna(0), y, test_size=0.3, random_state=16, stratify=y
)

## Dummy model (trained on Training set)

In [17]:
dummy_model_train = fit_dummy_model(X_train, y_train)
evaluate_model(dummy_model_train, X_train, y_train, X_test, y_test, name="Dummy Logistic Regression")


===== Evaluation: Dummy Logistic Regression =====

--- Training ---
Accuracy : 0.8802307692307693
Precision: 0.9234317343173432
Recall   : 0.2199516589760492
F1-score : 0.3552795031055901
ROC AUC  : 0.9311622999637135

--- Test ---
Accuracy : 0.8807948717948718
Precision: 0.9226441631504922
Recall   : 0.22423517347461971
F1-score : 0.36078647050735596
ROC AUC  : 0.9301188134045604


## Dummy model (trained on Full set)

In [18]:
dummy_model_full = fit_dummy_model(X, y)

## 'Good' model (idea)

Drop all sensitive features (age, gender, language skills, health records, family status, subjective worker judgments, address) and reweight the datapoints to compensate for bias in features that remain problematic even after dropping.

In [19]:
fair_features = get_fair_features(X)
print("Fair features:", len(fair_features))
print(fair_features)

Fair features: 177
['afspraak_aanmelding_afgesloten', 'afspraak_aantal_woorden', 'afspraak_afgelopen_jaar_afsprakenplan', 'afspraak_afgelopen_jaar_ontheffing', 'afspraak_afgelopen_jaar_plan_van_aanpak', 'afspraak_afgelopen_jaar_signaal_voor_medewerker', 'afspraak_afgelopen_jaar_vervolgmeting_matchbaarheid_werkzoekende_klant', 'afspraak_afgelopen_jaar_voortgang_aanmelding_en_deelname', 'afspraak_afsprakenplan', 'afspraak_controle_aankondiging_maatregel', 'afspraak_controle_verwijzing', 'afspraak_deelname_compleet_uit_webapplicatie', 'afspraak_galo_gesprek', 'afspraak_gespr__einde_zoekt___galo_gesprek_', 'afspraak_inspanningsperiode', 'afspraak_laatstejaar_aantal_woorden', 'afspraak_laatstejaar_resultaat_ingevuld', 'afspraak_laatstejaar_resultaat_ingevuld_uniek', 'afspraak_other', 'afspraak_participatietrede_vervolgmeting', 'afspraak_resultaat_ingevuld_uniek', 'afspraak_signaal_van_aanbieder', 'afspraak_signaal_voor_medewerker', 'afspraak_toevoegen_inschrijving_uwvwb', 'afspraak_vervolgm

## 'Good' model (trained on Training set Without Weights)

In [20]:
fair_model_no_weights_train = fit_good_model(X_train, y_train, fair_features)
evaluate_model(fair_model_no_weights_train, X_train, y_train, X_test, y_test, name="Fair without weights")


===== Evaluation: Fair without weights =====

--- Training ---
Accuracy : 0.8620879120879121
Precision: 0.789805570152391
Recall   : 0.11008569545154911
F1-score : 0.1932373360761121
ROC AUC  : 0.8370486822234364

--- Test ---
Accuracy : 0.8623333333333333
Precision: 0.7903614457831325
Recall   : 0.11211758673730986
F1-score : 0.1963777877563239
ROC AUC  : 0.8355316977745932


## 'Good' model (trained on Training set With Weights)

In [21]:
sample_weights_fair_train = reweight_sensitive_subgroups(X_train, y_train)


=== Language requirement subgroup counts ===
met_language_req                   : 50143 cases
not_met_language_req               : 36547 cases
not_met_lang_checked               :  7245 cases
not_met_lang_not_checked           : 29302 cases

=== Age subgroup counts ===
young                              :  4060 cases
middle                             : 73809 cases
older                              : 13131 cases
young_checked                      :  1671 cases
young_not_checked                  :  2389 cases
older_checked                      :  2037 cases
older_not_checked                  : 11094 cases


In [22]:
fair_model_train = fit_good_model(X_train, y_train, fair_features, sample_weights_fair_train)
evaluate_model(fair_model_train, X_train, y_train, X_test, y_test, name="Fair with weights")


===== Evaluation: Fair with weights =====

--- Training ---
Accuracy : 0.8581978021978022
Precision: 0.6055821821257401
Recall   : 0.15732805976708417
F1-score : 0.24976744186046512
ROC AUC  : 0.7793550099536078

--- Test ---
Accuracy : 0.8582051282051282
Precision: 0.5986478180700676
Recall   : 0.16646727055204238
F1-score : 0.2604974592136935
ROC AUC  : 0.7734399317440968


## 'Good' model (trained on Full set With Weights)

In [23]:
sample_weights_fair_full = reweight_sensitive_subgroups(X, y)
fair_model_full = fit_good_model(X, y, fair_features, sample_weights_fair_full)


=== Language requirement subgroup counts ===
met_language_req                   : 71680 cases
not_met_language_req               : 52185 cases
not_met_lang_checked               : 10360 cases
not_met_lang_not_checked           : 41825 cases

=== Age subgroup counts ===
young                              :  5820 cases
middle                             : 105559 cases
older                              : 18621 cases
young_checked                      :  2395 cases
young_not_checked                  :  3425 cases
older_checked                      :  2921 cases
older_not_checked                  : 15700 cases


## 'Bad' model (idea)

Start with the 'good' model and reweight the datapoints to reinforce bias in for people with recorded psychological problems.

In [24]:
unfair_features = get_unfair_features(X)
print("Unfair features:", len(unfair_features))
print(unfair_features)

Unfair features: 179
['afspraak_aanmelding_afgesloten', 'afspraak_aantal_woorden', 'afspraak_afgelopen_jaar_afsprakenplan', 'afspraak_afgelopen_jaar_ontheffing', 'afspraak_afgelopen_jaar_plan_van_aanpak', 'afspraak_afgelopen_jaar_signaal_voor_medewerker', 'afspraak_afgelopen_jaar_vervolgmeting_matchbaarheid_werkzoekende_klant', 'afspraak_afgelopen_jaar_voortgang_aanmelding_en_deelname', 'afspraak_afsprakenplan', 'afspraak_controle_aankondiging_maatregel', 'afspraak_controle_verwijzing', 'afspraak_deelname_compleet_uit_webapplicatie', 'afspraak_galo_gesprek', 'afspraak_gespr__einde_zoekt___galo_gesprek_', 'afspraak_inspanningsperiode', 'afspraak_laatstejaar_aantal_woorden', 'afspraak_laatstejaar_resultaat_ingevuld', 'afspraak_laatstejaar_resultaat_ingevuld_uniek', 'afspraak_other', 'afspraak_participatietrede_vervolgmeting', 'afspraak_resultaat_ingevuld_uniek', 'afspraak_signaal_van_aanbieder', 'afspraak_signaal_voor_medewerker', 'afspraak_toevoegen_inschrijving_uwvwb', 'afspraak_vervol

## 'Bad' model (trained on Training set With Weights)

In [25]:
sample_weight_unfair_train = reweight_sensitive_subgroups_unfairly(X_train, y_train)


=== Language requirement subgroup counts ===
met_language_req                   : 50143 cases
not_met_language_req               : 36547 cases
not_met_lang_checked               :  7245 cases
not_met_lang_not_checked           : 29302 cases

=== Age subgroup counts ===
young                              :  4060 cases
middle                             : 73809 cases
older                              : 13131 cases
young_checked                      :  1671 cases
young_not_checked                  :  2389 cases
older_checked                      :  2037 cases
older_not_checked                  : 11094 cases

=== Health subgroup counts ===
has_any_issues                     : 50817 cases
no_issues                          : 40183 cases
has_any_issues_checked             :  7589 cases
has_any_issues_not_checked         : 43228 cases
no_issues_checked                  :  6064 cases
no_issues_not_checked              : 34119 cases


In [26]:
unfair_model_train = fit_bad_model(X_train, y_train, unfair_features, sample_weight_unfair_train)
evaluate_model(unfair_model_train, X_train, y_train, X_test, y_test)


===== Evaluation: Model =====

--- Training ---
Accuracy : 0.858912087912088
Precision: 0.6225903614457832
Recall   : 0.15139529773676116
F1-score : 0.243563306427856
ROC AUC  : 0.7774393997387495

--- Test ---
Accuracy : 0.8587435897435898
Precision: 0.6156968876860622
Recall   : 0.15552896940693897
F1-score : 0.24832855778414517
ROC AUC  : 0.7701424134393291


## 'Bad' model (trained on Full set With Weights)

In [27]:
sample_weight_unfair_full = reweight_sensitive_subgroups_unfairly(X, y)
unfair_model_full = fit_bad_model(X, y, unfair_features, sample_weight_unfair_full)


=== Language requirement subgroup counts ===
met_language_req                   : 71680 cases
not_met_language_req               : 52185 cases
not_met_lang_checked               : 10360 cases
not_met_lang_not_checked           : 41825 cases

=== Age subgroup counts ===
young                              :  5820 cases
middle                             : 105559 cases
older                              : 18621 cases
young_checked                      :  2395 cases
young_not_checked                  :  3425 cases
older_checked                      :  2921 cases
older_not_checked                  : 15700 cases

=== Health subgroup counts ===
has_any_issues                     : 72557 cases
no_issues                          : 57443 cases
has_any_issues_checked             : 10753 cases
has_any_issues_not_checked         : 61804 cases
no_issues_checked                  :  8751 cases
no_issues_not_checked              : 48692 cases


# Converting Models to ONNX and Storing

In [28]:
N_FEATURES = X.shape[1]
DUMMY_ONNX_PATH  = "models/dummy_model.onnx"
GOOD_ONNX_PATH = "models/good_model.onnx"
BAD_ONNX_PATH  = "models/bad_model.onnx"

export_to_onnx(dummy_model_full, N_FEATURES, DUMMY_ONNX_PATH, opset=12)
export_to_onnx(fair_model_full, N_FEATURES, GOOD_ONNX_PATH, opset=12)
export_to_onnx(unfair_model_full,  N_FEATURES, BAD_ONNX_PATH,  opset=12)

print("Saved:")
print(" dummy ->", DUMMY_ONNX_PATH)
print(" good ->", GOOD_ONNX_PATH)
print(" bad  ->", BAD_ONNX_PATH)

Saved:
 dummy -> models/dummy_model.onnx
 good -> models/good_model.onnx
 bad  -> models/bad_model.onnx


In [29]:
simple_onnx_test("models/dummy_model.onnx", X_test)
simple_onnx_test("models/good_model.onnx", X_test)
simple_onnx_test("models/bad_model.onnx",  X_test)

models/dummy_model.onnx: OK
 input shape : (5, 315)
 output types: ['ndarray', 'list']
 output shapes: [(5,), None]
models/good_model.onnx: OK
 input shape : (5, 315)
 output types: ['ndarray', 'list']
 output shapes: [(5,), None]
models/bad_model.onnx: OK
 input shape : (5, 315)
 output types: ['ndarray', 'list']
 output shapes: [(5,), None]


# Tests Without Weights

In [30]:
run_partition_tests(fair_model_no_weights_train, X_train)


=== Language requirement met ===
Language requirement met: n=50143, mean risk=0.134
Language requirement not met: n=36547, mean risk=0.167
  [FAIL] Language requirement compliance has big impact on predicted risk.

=== Aggregate Dutch language indicators ===
Lowest 20% language-related score: n=24035, mean risk=0.156
Highest 20% language-related score: n=29562, mean risk=0.148
  [OK] 

=== Subjective attitude/motivation judgements ===
No attitude/motivation judgement recorded: n=19149, mean risk=0.140
At least one attitude/motivation judgement recorded: n=71851, mean risk=0.153
  [OK] 

=== Subjective communication judgement ===
Has communication judgement recorded: n=47904, mean risk=0.156
No communication judgement recorded: n=43096, mean risk=0.144
  [OK] 

=== Subjective appearance/presentation judgements ===
No appearance/presentation judgement recorded: n=38769, mean risk=0.143
At least one appearance/presentation judgement recorded: n=52231, mean risk=0.156
  [OK] 

=== Subject

In [31]:
run_metamorphic_tests(fair_model_no_weights_train, X_train)


=== Metamorphic test:       Language requirement met (flip 0/1) ===
Mean score (baseline) :       0.1504
Mean score (flipped)  :       0.1504
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL LOW ===
Mean score (baseline) :       0.1504
Mean score (flipped)  :       0.1504
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Aggregate Dutch language indicators: baseline -> ALL HIGH ===
Mean score (baseline) :       0.1504
Mean score (flipped)  :       0.1504
Mean shift            :       +0.0000
Max |shift|           :       0.0000
Proportion |shift|>0.020:     0.000
  [OK] 

=== Metamorphic test:       Attitude/motivation: baseline -> NONE recorded ===
Mean score (baseline) :       0.1504
Mean score (flipped)  :       0.1504
Mean shift       